# Chapter 1 - Introducing Token Factory

**AI Agent Course** · Nebius Token Factory × NVIDIA

**Goal:** By the end of the chapter, you can use Token Factory to select and call a model from Python, handle one tool request, return the tool result to the model, and inspect token usage.

**Prerequisites:** Python 3.10+, basic familiarity with REST APIs. No GPU, no prior LLM experience assumed.

**Deliverable:** a small Python script that calls Nemotron 3 Ultra, asks it to emit a structured JSON tool call, executes the tool locally, and returns the result to the model - this is the basis of the agent we scale in the next chapters.

## 1. What is Token Factory?

Token Factory is **managed inference for open-source models**. You don't manage GPUs or serving stacks - Token Factory handles inference for you.

**In this course, we use serverless inference - public models available to all users.**

### The model catalog

The catalog hosts frontier open models: **GLM 5.2**, **Kimi 3** and the **NVIDIA Nemotron family**, among others. Our flagship models for this course are the **Nvidia Nemotron** series — we will explore a few of these models and make decisions on which is the best for us.

### Why "OpenAI-compatible API" matters

Token Factory exposes an OpenAI-compatible API. That means:

- The entire ecosystem of SDKs and tools works out of the box
- Drop-in migration from other providers
- Every tutorial on the internet works with a **one-line `base_url` change**

## 2. Setup: account, API key, and environment

1. Go to [tokenfactory.nebius.com](https://tokenfactory.nebius.com) and create an account.
2. Open **Get API Key → Create API key** and copy it (you can't view it again later).
3. Nebius exposes an **OpenAI-compatible** API, so we drive it with the plain `openai` SDK.

![Token Factory home](images/token-factory-home.png)
![API Key creation](images/api-key-creation.png)

Your `lesson/.env` should look like:

```
NEBIUS_API_KEY=...
```

**Key hygiene**: keys live in `.env`, never in code, and `.env` goes in `.gitignore`. Never commit a notebook with a real key in it unless you want someone stealing your Token Factory credits!

In [1]:
%pip install -q openai python-dotenv rich sympy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

from dotenv import load_dotenv
from rich import print

load_dotenv("lesson/.env")
%load_ext rich

assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in lesson/.env"
print("Keys loaded.")

The rich extension is already loaded. To reload it, use:
  %reload_ext rich


Keys loaded.

## 3. The model catalog

The model catalog is the home for all models on Token Factory. Here, you can view basic information, and types of availability.

![Model catalog](images/model-catalog.png)


## 4. Reading a model card

If you were deciding on an engine to be put in a car, you would likely read through different spec sheets until you found the one to fit your needs.

Every model in the catalog has a card - this is its spec sheet. The model cards are something you will come back to constantly: for the model string, for prices, for context limits, for what the model can and can't do. Need a fast car? You will pick the strongest - and likely most expensive - option. If you care about efficiency, you would choose one that is smaller and cheaper. The same choices will persist for the models you choose.

In this course, we'll use **Nemotron Class Models** as a running example.

Click into **Nemotron-3-Ultra-550b-a55b**, scroll down, and click the model card.

![Nemotron 3 Ultra model card](images/model-card.png)

### Copying the model string

At the top of the endpoint properties you'll find the **routing key** — for our model it's `nvidia/Nemotron-3-Ultra-550b-a55b`, with a copy icon right next to it. **That exact string is what you pass as `model=` in every API call.** Always copy it from the card rather than typing it; a typo here is the single most common first error, and the API's "model not found" message won't tell you which character you got wrong.

The name itself encodes useful information: `550b` is the total parameter count, `a55b` means ~55B parameters are *active* per token — this is a **hybrid MoE (Mixture of Experts)** model, which is how it can be both huge and reasonably fast.

### Reading the model card - what specs are important to us?


**Context window — 1,024K tokens.**
This is the ceiling on everything the model can see at once: your prompt, the conversation history, any documents you paste in, *and* the output it generates. They all share this budget. Think of it as the model's working memory. A bigger window means you can hand it an entire codebase or a stack of contracts in one go; a smaller one means you'll have to chop things up and manage what the model gets to see. That management problem is real engineering, and it's exactly what the memory/context chapter is about later in the course.

**Pricing — \$1.00 / 1M input tokens, \$3.00 / 1M output tokens.**
Pricing is in terms of tokens, and there is a reason for the separation. Input tokens are cheap because the model ingests your whole prompt in a single go; output tokens cost more because the model generates them one at a time. Each one is a full trip through the model. Practical consequence: a chatty model that writes long answers costs you much more (think, what can we do to outputs to reduce this?).

**Modality — Text-to-text.**
What goes in and what comes out. Our model reads text and writes text, but scroll through the catalog and you'll see vision models (they accept images) and embedding models (they output vectors, not sentences). Checking this field takes two seconds and saves you from building half a project around a model that can't do what you assumed.

**Quantization — FP4.**
The numeric precision the model's weights are served at. Models are trained at high precision, then compressed to run faster and cheaper — like streaming a movie at a lower bitrate. You don't need to make any decisions about this in this course; just know what the label means when you see it, and that it's part of why serving a 550B-parameter model is economically possible at all.

**Tool calling / Reasoning — Available.**
These flags tell you what the model was *fine-tuned* to do, beyond plain chat. "Tool calling: available" means the model was trained to emit well-formed function calls natively. Keep this in mind for later chapters. "Reasoning: available" means the model can spend extra tokens thinking before answering, which helps on hard problems and costs you on easy ones. If either flag is missing from a card, how do you think that will impact agent work?

**License.**
Open-weight does not mean do-whatever-you-want. Licenses differ on commercial use, redistribution, and what you can train on the outputs. Important to understand before putting a model in production, or starting work on something you may not be allowed to do.

### Exercise: pick a model by reading its card

Open two cards side by side: **`nvidia/Nemotron-3-Ultra-550b-a55b`** and **`nvidia/nemotron-3-super-120b-a12b`**.

For each workload below, decide: **which one or two card fields settle the decision, and which of the two models do you pick?** Write one sentence of justification per workload — "it's bigger" doesn't count.

1. **A chat summarizer** — condenses customer support conversations into two sentences, thousands of times per day.
2. **A high-volume classification workload** — labels incoming tickets as `billing` / `bug` / `feature request`, millions per month.
3. **A contract analyst** — answers questions about 300-page legal documents pasted in whole.
4. **An agent that books travel** — must emit function calls to search flights and hotels, and plan multi-step itineraries at high reliability.

## 5. First calls in the UI (Playground)

Before any code, use the **Token Factory playground**:

1. Open the model card for **Nemotron 3 Ultra** and click **Go to playground**.
2. Send some input to the model, and observe its behavior.
3. Look at the **Metrics below the response**. Observe the numbers you see (it is OK to not understand these yet).
4. **Eyeball-test** the same prompt on **Nemotron 3 Super** side by side.

Make observations about how the models differ using the context of the situations above.

## 6. The Chat Completions request

A Chat Completions request is the standard way to interact with models via API. It is a structured format for how we communicate to models and how they communicate back.

A sample completions request for your model can be seen on the model card. 

![Chat Completions example](images/model-card-completions.png)


### 6.1 Anatomy of the request

Before the code, the pieces:

- **`model`** *(required)* — the routing key you copied from the card.
- **`messages`** *(required)* — the conversation as a list of roles. **Key note: the API is stateless — you send the whole conversation history every time.** The roles:
  - **`system`** — instructions for how the model should behave ("You are a concise assistant..."). Set once, at the top.
  - **`user`** — what the human says. Your prompts go here!
  - **`assistant`** — what the model said. You append its previous replies here so it remembers the conversation.
  - **`tool`** — results from function calls, which we'll meet in the next chapter.
- **Sampling params** *(optional)* — `temperature`, `top_p`, `max_tokens`. We will not worry about these for now.
- **The response object** — what comes back: `choices[0].message.content` (the answer), `finish_reason` (why it stopped), and crucially for this course, **`usage`** (prompt/completion tokens), which is our cost signal.

### 6.2 Minimal call

The model card provides a ready-made snippet ("Use model in code" — the one below is adapted from it), so you never have to write this boilerplate from memory:

In [3]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.us-central1.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "In two sentences, what does NVIDIA's Nemotron model family target?"

response = client.chat.completions.create(
    model="nvidia/Nemotron-3-Ultra-550b-a55b",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

print(response.choices[0].message.content)
print("---")
print("finish_reason:", response.choices[0].finish_reason)
print("usage:", response.usage)

NVIDIA’s Nemotron model family targets high‑performance, enterprise‑grade language models optimized for inference 
speed and scalability on NVIDIA GPUs. They aim to deliver state‑of‑the‑art natural‑language understanding and 
generation for commercial AI applications such as chatbots, content creation, and code assistance.

---

finish_reason: stop

usage:
CompletionUsage(
    completion_tokens=106,
    prompt_tokens=152,
    total_tokens=258,
    completion_tokens_details=CompletionTokensDetails(
        accepted_prediction_tokens=None,
        audio_tokens=None,
        reasoning_tokens=26,
        rejected_prediction_tokens=None
    ),
    prompt_tokens_details=None,
    prompt_cache_hit_tokens=0,
    prompt_cache_miss_tokens=152
)

### Exercise: try to create a request that does these tasks!

#### 1. Only produce 10 tokens or less per response

In [5]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.us-central1.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "In two sentences, what does NVIDIA's Nemotron model family target?"

response = client.chat.completions.create(
    model="nvidia/Nemotron-3-Ultra-550b-a55b",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
    max_tokens=10,
)

print(response.choices[0].message.content)
print("---")
print("finish_reason:", response.choices[0].finish_reason)
print("usage:", response.usage)

The user asks: "In two sentences, what

---

finish_reason: length

usage:
CompletionUsage(
    completion_tokens=10,
    prompt_tokens=152,
    total_tokens=162,
    completion_tokens_details=CompletionTokensDetails(
        accepted_prediction_tokens=None,
        audio_tokens=None,
        reasoning_tokens=7,
        rejected_prediction_tokens=None
    ),
    prompt_tokens_details=None,
    prompt_cache_hit_tokens=0,
    prompt_cache_miss_tokens=152
)

#### 2. Set a system prompt to make the model an expert in the Nemotron model family

In [6]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.us-central1.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

SYSTEM_PROMPT = "You are an expert in the Nemotron family of models. You know all about their capabilities and applications, as well as the current state of their development. You are concise and helpful to the user."
USER_MESSAGE = "In two sentences, what does NVIDIA's Nemotron model family target?"

response = client.chat.completions.create(
    model="nvidia/Nemotron-3-Ultra-550b-a55b",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

print(response.choices[0].message.content)
print("---")
print("finish_reason:", response.choices[0].finish_reason)
print("usage:", response.usage)

NVIDIA’s Nemotron model family targets high‑performance, scalable language understanding and generation for 
enterprise‑grade AI applications. It aims to deliver state‑of‑the‑art accuracy while optimizing inference 
efficiency across diverse hardware platforms.

---

finish_reason: stop

usage:
CompletionUsage(
    completion_tokens=98,
    prompt_tokens=180,
    total_tokens=278,
    completion_tokens_details=CompletionTokensDetails(
        accepted_prediction_tokens=None,
        audio_tokens=None,
        reasoning_tokens=36,
        rejected_prediction_tokens=None
    ),
    prompt_tokens_details=None,
    prompt_cache_hit_tokens=0,
    prompt_cache_miss_tokens=180
)

#### 3. Challenge! Make the model produce 10 identical responses with:

```python
SYSTEM_PROMPT = "You are an expert barista."
USER_MESSAGE = "Produce 5 bullet points on why an Americano is the best coffee drink."
```

**Hint: you may need to use a loop and a certain parameter that makes the model more deterministic (less creative)**

In [11]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.us-central1.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

SYSTEM_PROMPT = "You are an expert barista."
USER_MESSAGE = "Produce 5 bullet points on why an Americano is the best coffee drink."

response = client.chat.completions.create(
    model="nvidia/Nemotron-3-Ultra-550b-a55b",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT}, ## What goes here?
        {"role": "user", "content": USER_MESSAGE},
    ],
    ## What goes here?
    temperature=0,
)

## What goes here?
for i in range (9):
    print(response.choices[0].message.content)
    print("---")
    print("finish_reason:", response.choices[0].finish_reason)
    print("usage:", response.usage)

- **Balanced Flavor Profile** – The Americano blends a smooth espresso base with just the right amount of hot 
water, delivering a rich, full‑bodied taste without the bitterness of a straight shot.  
- **Customizable Strength** – By adjusting the water‑to‑espresso ratio, you can tailor the drink from a light, 
tea‑like sip to a bold, near‑espresso intensity, suiting any palate.  
- **Lower Acidity** – Diluting espresso with hot water softens the natural acidity, making it gentler on the 
stomach while preserving the coffee’s aromatic complexity.  
- **Versatile Temperature** – Served hot, it’s a comforting morning ritual; served over ice, it becomes a 
refreshing, low‑calorie iced coffee perfect for warm afternoons.  
- **Simplicity & Consistency** – With only two ingredients (espresso + water), the Americano is easy to replicate 
at home or in any café, ensuring a reliably excellent cup every time.

---

finish_reason: stop

usage:
CompletionUsage(
    completion_tokens=271,
    prompt_tokens=148,
    total_tokens=419,
    completion_tokens_details=CompletionTokensDetails(
        accepted_prediction_tokens=None,
        audio_tokens=None,
        reasoning_tokens=39,
        rejected_prediction_tokens=None
    ),
    prompt_tokens_details=None,
    prompt_cache_hit_tokens=0,
    prompt_cache_miss_tokens=148
)

- **Balanced Flavor Profile** – The Americano blends a smooth espresso base with just the right amount of hot 
water, delivering a rich, full‑bodied taste without the bitterness of a straight shot.  
- **Customizable Strength** – By adjusting the water‑to‑espresso ratio, you can tailor the drink from a light, 
tea‑like sip to a bold, near‑espresso intensity, suiting any palate.  
- **Lower Acidity** – Diluting espresso with hot water softens the natural acidity, making it gentler on the 
stomach while preserving the coffee’s aromatic complexity.  
- **Versatile Temperature** – Served hot, it’s a comforting morning ritual; served over ice, it becomes a 
refreshing, low‑calorie iced coffee perfect for warm afternoons.  
- **Simplicity & Consistency** – With only two ingredients (espresso + water), the Americano is easy to replicate 
at home or in any café, ensuring a reliably excellent cup every time.

...

usage:
CompletionUsage(
    completion_tokens=271,
    prompt_tokens=148,
    total_tokens=419,
    completion_tokens_details=CompletionTokensDetails(
        accepted_prediction_tokens=None,
        audio_tokens=None,
        reasoning_tokens=39,
        rejected_prediction_tokens=None
    ),
    prompt_tokens_details=None,
    prompt_cache_hit_tokens=0,
    prompt_cache_miss_tokens=148
)

- **Balanced Flavor Profile** – The Americano blends a smooth espresso base with just the right amount of hot 
water, delivering a rich, full‑bodied taste without the bitterness of a straight shot.  
- **Customizable Strength** – By adjusting the water‑to‑espresso ratio, you can tailor the drink from a light, 
tea‑like sip to a bold, near‑espresso intensity, suiting any palate.  
- **Lower Acidity** – Diluting espresso with hot water softens the natural acidity, making it gentler on the 
stomach while preserving the coffee’s aromatic complexity.  
- **Versatile Temperature** – Served hot, it’s a comforting morning ritual; served over ice, it becomes a 
refreshing, low‑calorie iced coffee perfect for warm afternoons.  
- **Simplicity & Consistency** – With only two ingredients (espresso + water), the Americano is easy to replicate 
at home or in any café, ensuring a reliably excellent cup every time.

---

finish_reason: stop

usage:
CompletionUsage(
    completion_tokens=271,
    prompt_tokens=148,
    total_tokens=419,
    completion_tokens_details=CompletionTokensDetails(
        accepted_prediction_tokens=None,
        audio_tokens=None,
        reasoning_tokens=39,
        rejected_prediction_tokens=None
    ),
    prompt_tokens_details=None,
    prompt_cache_hit_tokens=0,
    prompt_cache_miss_tokens=148
)

### 6.3 Streaming

Contrary to what you may be used to, you would notice that the responses came out as one block instead of a streaming - 'typed looking' - response. Note that we can control this behavior, and get our response as a growing block of text.

In [13]:
stream = client.chat.completions.create(
    model="nvidia/Nemotron-3-Ultra-550b-a55b",
    messages=[{"role": "user", "content": "What is LLM streaming?"}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

**

LLM streaming** refers to

the process

of delivering

a

...

feed,

improving

responsiveness and usability

for interactive

applications.

## 7. A very simple tool call

**What is it?:** a *tool* is just a function; a *tool call* is just the model asking us to run it. In agents, this is how we execute actions. An agent could query a database, fetch some document, or call an API. Each of these are functions we can request our model to run.

### 7.1 First contact: tool calling in the model card UI

Before writing any code:

1. Open the **Nemotron 3 Ultra** model card playground and find the **Functions and JSON output** dropdown. Click **+ Add function**.

![Function dropdown](images/model-card-tool-call.png)

2. Paste this function into the UI and click **Add function**.

```json
{
  "name": "get_weather",
  "description": "Get current weather for a city",
  "parameters": {
    "type": "object",
    "properties": {
      "city": {"type": "string"}
    },
    "required": ["city"]
  }
}
```

![Tool call paste](images/tool-call-paste.png)

3. Ask **"What's the weather in Atlanta?"** and watch the model respond with a *tool call* instead of an answer.

Notice what the UI shows: the function name, the parsed arguments, and the fact that **nothing was executed** — the model only produced a *request*. Someone (in a minute: our Python code) has to run the function and hand the result back.


![Tool call in the playground](images/tool-call-success.png)

Now that you've seen it, let's build one that can actually execute, from scratch.

### 7.2 Manual JSON tool call

When you ask an LLM to calculate a math equation, does it actually do computation? At the basic level, not really. It is familiar with mathematical concepts and expressions from its training, so it outputs information it *knows* rather than what it verifies by crunching numbers.

Let's create a **calculator tool** in Python that our LLM can actually use, so its answers are grounded in computation.

### Here is our tool, which uses SymPy to parse and evaluate math expressions.

In [14]:

def calculate(expression: str) -> str:
    return str(sympy.sympify(expression))

#### Now, we will use the tool we have created.

In [15]:
import json
import sympy

TOOL_SYSTEM_PROMPT = """You can use one tool:
calculate(expression: str) -> the exact result of an arithmetic expression
If the user's request needs the tool, respond with ONLY this JSON, no other text:
{"tool": "calculate", "arguments": {"expression": "<expression>"}}
Otherwise, answer normally."""


messages = [
    {"role": "system", "content": TOOL_SYSTEM_PROMPT},
    {"role": "user", "content": "What's 4823 * 3917?"},
]

resp = client.chat.completions.create(
    model="nvidia/Nemotron-3-Ultra-550b-a55b",
    messages=messages,
    temperature=0,
)
raw = resp.choices[0].message.content
print("Model said:", raw)

call = json.loads(raw)
result = calculate(**call["arguments"])

# Feed the result back for a final answer
messages.append({"role": "assistant", "content": raw})
messages.append({"role": "user", "content": f"Tool result: {result}. Answer the user."})

final = client.chat.completions.create(
    model="nvidia/Nemotron-3-Ultra-550b-a55b",
    messages=messages,
)
print(final.choices[0].message.content)

Model said: 
{"tool": "calculate", "arguments": {"expression": "4823 * 3917"}}

4823 * 3917 = 18,891,691

### Exercise: Break it!

The code above works well - our tool call is simple and we can expect it to do as we asked. But what can go wrong here, especially when we have less view into what our model is and needs to do? 

Your job: **make the loop crash (or misbehave) in at least three different ways**, using nothing but the user message.

Hints:
1. What should the model be able and not be able to do?
2. What if our inputs differ from what the function expects?
3. Inspect the tool itself - what can it do and what can't it do?

For each break you find, write one line: **what the model emitted → which line of our code died → whose fault it really was.**

In [ ]:
import json
import sympy

TOOL_SYSTEM_PROMPT = """You can use one tool:
calculate(expression: str) -> the exact result of an arithmetic expression
If the user's request needs the tool, respond with ONLY this JSON, no other text:
{"tool": "calculate", "arguments": {"expression": "<expression>"}}
Otherwise, answer normally."""


messages = [
    {"role": "system", "content": TOOL_SYSTEM_PROMPT},
    {"role": "user", "content": ""}, ## What goes here?
]

resp = client.chat.completions.create(
    model="nvidia/Nemotron-3-Ultra-550b-a55b",
    messages=messages,
    temperature=0,
)
raw = resp.choices[0].message.content
print("Model said:", raw)

call = json.loads(raw)
result = calculate(**call["arguments"])

# Feed the result back for a final answer
messages.append({"role": "assistant", "content": raw})
messages.append({"role": "user", "content": f"Tool result: {result}. Answer the user."})

final = client.chat.completions.create(
    model="nvidia/Nemotron-3-Ultra-550b-a55b",
    messages=messages,
)
print(final.choices[0].message.content)

Model said: There is no variable **x** in the equation `5 = 3 + 4`. The equation simply states that 5 equals 3 plus
4, which is false because 3 + 4 = 7. If you meant to ask for **x** in an equation like `5 = 3 + x`, then **x = 2**.

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:24                                                                                   │
│                                                                                                  │
│   21 raw = resp.choices[0].message.content                                                       │
│   22 print("Model said:", raw)                                                                   │
│   23                                                                                             │
│ ❱ 24 call = json.loads(raw)                                                                      │
│   25 result = calculate(**call["arguments"])                                                     │
│   26                                                                                             │
│   27 # Feed the result back for a final answer                                                   │
│                                                                                                  │
│ /Users/alexhanley/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/json/ │
│ __init__.py:346 in loads                                                                         │
│                                                                                                  │
│   343 │   if (cls is None and object_hook is None and                                            │
│   344 │   │   │   parse_int is None and parse_float is None and                                  │
│   345 │   │   │   parse_constant is None and object_pairs_hook is None and not kw):              │
│ ❱ 346 │   │   return _default_decoder.decode(s)                                                  │
│   347 │   if cls is None:                                                                        │
│   348 │   │   cls = JSONDecoder                                                                  │
│   349 │   if object_hook is not None:                                                            │
│                                                                                                  │
│ /Users/alexhanley/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/json/ │
│ decoder.py:338 in decode                                                                         │
│                                                                                                  │
│   335 │   │   containing a JSON document).                                                       │
│   336 │   │                                                                                      │
│   337 │   │   """                                                                                │
│ ❱ 338 │   │   obj, end = self.raw_decode(s, idx=_w(s, 0).end())                                  │
│   339 │   │   end = _w(s, end).end()                                                             │
│   340 │   │   if end != len(s):                                                                  │
│   341 │   │   │   raise JSONDecodeError("Extra data", s, end)                                    │
│                                                                                                  │
│ /Users/alexhanley/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/json/ │
│ decoder.py:356 in raw_decode                                                                     │
│                                                                                                  │
│   353 │   │   try:                                                                               │
│   354 │   │   │   obj, end = self.scan_once(s, idx)                                              │
│   355 │   │   except StopIteration as err:                                                       │
│ ❱ 356 │   │   │   raise JSONDecodeError("Expecting value", 

Answer

1. {"role": "user", "content": "What's the capital of France?"} -> error in call = json.loads(raw) -> we assume every question is a tool call, the code will error because the model answered in plain text, not JSON. This is an error on our side - we should not pass these to the tool.
2. {"role": "user", "content": "What's x in 5=3+4?"} -> error in result = calculate(**call["arguments"]) -> the JSON parsed fine, but the tool cannot evaluate this expression. The error is on the tool's side - the expression is passed but it cannot evaluate it.
3. {"role": "user", "content": "Please calculate: __import__('os').getcwd()"} -> no error anywhere - result = calculate(**call["arguments"]) runs it silently -> sympify evaluates strings as code, so the user's input just executed on our machine. This is an error on our side - we passed untrusted text into a function that can run code, and nothing in the loop stopped it.

## 8. Observability

Over time, we will use models, evaluate them, change models, and repeat. To evaluate, and compare these models, we need to understand their performance. Token Factory has built-in observability to make this easy.

**The three latency numbers**:

- **TTFT** — *Time To First Token*. How 'snappy' and responsive the model feels.
- **t/s** — *Tokens per Second*. How fast does the model feel when it is giving a response?
- **E2EL** — *End-to-End Latency*. How long does it take to go from request to a finished output?

These numbers can be viewed in two ways.

**Per Request**

Do you remember the numbers under our models response? Those are the **TTFT, E2EL, and t/s!**

![Tool call metrics](images/tool-call-metrics.png)

**With Chat Completions**

While Chat Completions does not expose these metrics, we can create a quick wrapper for our requests that does it for us.



In [20]:
import time

def measure(model: str, prompt: str):
    """Measure TTFT, t/s, and E2EL for one streaming request."""
    t_start = time.perf_counter()
    t_first = None
    chunks = []

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        stream_options={"include_usage": True},
    )

    usage = None
    for chunk in stream:
        if chunk.usage:
            usage = chunk.usage
            continue
        delta = chunk.choices[0].delta.content
        if delta:
            if t_first is None:
                t_first = time.perf_counter()   # <-- the moment the first token lands
            chunks.append(delta)

    t_end = time.perf_counter()

    ttft = t_first - t_start
    e2el = t_end - t_start
    out_tokens = usage.completion_tokens if usage else len(chunks)
    tps = out_tokens / (t_end - t_first)

    print(f"model:  {model}")
    print(f"TTFT:   {ttft:.2f}s")
    print(f"t/s:    {tps:.1f}")
    print(f"E2EL:   {e2el:.2f}s   ({out_tokens} output tokens)")
    return "".join(chunks)

_ = measure(
    "nvidia/Nemotron-3-Ultra-550b-a55b",
    "Explain what an AI agent is in one paragraph.",
)

model:  nvidia/Nemotron-3-Ultra-550b-a55b

TTFT:   0.61s

t/s:    533.7

E2EL:   0.78s   (95 output tokens)

### Exercise: Using the information below, manually calculate t/s and TTFT!

- 350 output tokens
- E2EL = 5.4s
- Time from TTFT to finishing = 3.1s



Answer:

TTFT = E2EL − generation time = 5.4s − 3.1s = **2.3s**

t/s = output tokens ÷ generation time = 350 ÷ 3.1s = **~113 t/s**

## 9. Costs and cost management

**LLMs can get expensive!** Every time the model is receiving input and producing output, it is actively billing. This is why cost management becomes **extremely important as our model scales**. Later, we will discuss best practices, so our agent does not break the bank.

The math itself is simple - the card gives you two prices, the response gives you two token counts:

$$\text{cost} = \frac{\text{prompt tokens} \times \text{price}_{in} + \text{completion tokens} \times \text{price}_{out}}{1{,}000{,}000}$$

A prototype request that costs a fraction of a cent feels free. The same request at 100,000 calls/day is a lot of money! When agents are running wild, we have less control over how much they use. Therefore, it is a great habit to **know what every request costs.**

Here is a wrapper we can use in our production system that calculates cost per request:

In [21]:
# Prices from the Nemotron 3 Ultra model card, $/1M tokens

PRICE_IN = 1.00
PRICE_OUT = 3.00

def tracked_completion(**kwargs):
    """Run a chat completion and print what it cost."""
    resp = client.chat.completions.create(**kwargs)
    u = resp.usage
    cost = (u.prompt_tokens * PRICE_IN + u.completion_tokens * PRICE_OUT) / 1e6
    print(f"input tokens:  {u.prompt_tokens}")
    print(f"output tokens: {u.completion_tokens}")
    print(f"cost:          ${cost:.6f}")
    return resp, cost

resp, cost = tracked_completion(
    model="nvidia/Nemotron-3-Ultra-550b-a55b",
    messages=[{"role": "user", "content": "Explain what an AI agent is in one paragraph."}],
)

input tokens:  135

output tokens: 173

cost:          $0.000654

#### One request seems cheap, but what about when we scale?

In [22]:
# 1 a day? 1,000? 100,000? 1,000,000?:
for requests_per_day in [1, 1_000, 100_000, 1_000_000]:
    print(f"{requests_per_day:>9,} requests/day  →  ${cost * requests_per_day:>10,.2f}/day  →  ${cost * requests_per_day * 30:>12,.2f}/month")

1 requests/day  →  $      0.00/day  →  $        0.02/month

1,000 requests/day  →  $      0.65/day  →  $       19.62/month

100,000 requests/day  →  $     65.40/day  →  $    1,962.00/month

1,000,000 requests/day  →  $    654.00/day  →  $   19,620.00/month

### Exercise: How is our bank feeling?

Calculate the cost per request for 2 models using the request above ( messages=[{"role": "user", "content": "Explain what an AI agent is in one paragraph."}],  ):
1. nvidia/Nemotron-3-Ultra-550b-a55b
2. nvidia/Nemotron-3-Nano-30B-A3B (you will need to find its pricing!)

What is the average cost per request? Per 1,000 requests? Per 100,000?

Answer:



In [ ]:
# Prices from the Nemotron 3 Ultra model card, $/1M tokens

PRICE_IN = 1.00
PRICE_OUT = 3.00

def tracked_completion(**kwargs):
    """Run a chat completion and print what it cost."""
    resp = client.chat.completions.create(**kwargs)
    u = resp.usage
    cost_model1 = (u.prompt_tokens * PRICE_IN + u.completion_tokens * PRICE_OUT) / 1e6
    print(f"input tokens:  {u.prompt_tokens}")
    print(f"output tokens: {u.completion_tokens}")
    print(f"cost:          ${cost_model1:.6f}")
    return resp, cost_model1

resp, cost_model1 = tracked_completion(
    model="nvidia/Nemotron-3-Ultra-550b-a55b",
    messages=[{"role": "user", "content": "Explain what an AI agent is in one paragraph."}],
)

# Prices from the Nemotron 3 Nano model card, $/1M tokens

PRICE_IN = 0.06
PRICE_OUT = 0.24

def tracked_completion(**kwargs):
    """Run a chat completion and print what it cost."""
    resp = client.chat.completions.create(**kwargs)
    u = resp.usage
    cost_model2 = (u.prompt_tokens * PRICE_IN + u.completion_tokens * PRICE_OUT) / 1e6
    print(f"input tokens:  {u.prompt_tokens}")
    print(f"output tokens: {u.completion_tokens}")
    print(f"cost:          ${cost_model2:.6f}")
    return resp, cost_model2

resp, cost_model2 = tracked_completion(
    model="nvidia/Nemotron-3-Ultra-550b-a55b",
    messages=[{"role": "user", "content": "Explain what an AI agent is in one paragraph."}],
)

print(f"Model 1 cost: ${cost_model1:.6f}")
print(f"Model 2 cost: ${cost_model2:.6f}")
print(f"Model 1 is {cost_model1 / cost_model2:.1f}x more expensive than Model 2 per request.")

input tokens:  135

output tokens: 111

cost:          $0.000468

input tokens:  135

output tokens: 146

cost:          $0.000043

Model 1 cost: $0.000468

Model 2 cost: $0.000043

Model 1 is 10.8x more expensive than Model 2 per this request.

### Watching spend in the Token Factory console

You don't have to track everything yourself — Token Factory records every billable token. To pull up your own spend:

1. In the **left-hand panel**, open **Billing settings → Usage**.

![Usage button](images/usage-button.png)

2. Switch the view to **Chart**.
3. Set **Group by → Products**.
4. Open the **Products** dropdown and select **`Nemotron-3-Ultra-550B-a55B Input`** and **`Nemotron-3-Ultra-550B-a55B Output`**.

![Billing dropdown](images/billing-dropdown.png)

Each day's bar now splits into the two products you selected:

![Daily spend for Nemotron 3 Ultra, input vs output](images/billing-graph.png)

Two things to notice in this chart:

- **Input and output are separate products** — because they're billed at separate rates, exactly as the model card said. And look at the proportions: the blue (output) portion dwarfs the green (input) on every bar. Output tokens cost 3× more, so even modest generations dominate costs. Capping `max_tokens` pays off right here.
- **The total up top is the ground truth.** Your `tracked_completion` estimates should agree with this page.

If you've been running this notebook, today's bar should include every call you've made this chapter. Two habits from day one: check this chart **per key** (each project gets its own key, so spend is attributable), and **set a spend limit per key** so a runaway loop can't burn the budget — that stops being hypothetical the moment agents start calling themselves in the coming chapters!

## 10. Wrap-up

You started this chapter with an account and nothing else. Here's what you can do now:

- **Read a model card like a spec sheet** — routing key, context window, pricing, quantization, tool-calling support — and make decisions based on workloads.

- **Call a model from Python** through the OpenAI-compatible API: minimal calls, streaming, and more.

- **Make a model use a tool.** You learned how models execute tool calls, and what can break them.

- **Measure what you run.** TTFT, t/s, and E2EL; cost per request from `usage` and two prices; ground truth in the billing console.

If one idea should stick, it's this: **treat every model call as a priced, measurable transaction.** Tokens in, tokens out, with a dollar figure and a latency number on every one. Agents are going to make *thousands* of these calls on your behalf — the habits from this chapter (log everything, know your costs) are what keep that from becoming a very expensive surprise.

**Where we're headed:** our calculator worked, but our weather tool was fake — the model asked for data nobody could fetch. Next chapter we plug in the first *real* tool: **web search via Tavily**. Combined with the loop you just built by hand, that gives us the first genuine agentic loop — a model that can decide it doesn't know something, go find out, and come back with an answer.

*See you in Chapter 2!* 